# 05. Red Flag Detection

**v6 Section 18.5** | Phase B 실패 패턴 자동 감지

## Red Flags (하나라도 해당 → 즉시 중단)

| # | Red Flag | 의미 | Phase B 사례 |
|---|----------|------|-----------|
| 1 | z 내부 uniform | 평면파로 수렴 = 회절 미학습 | Phase B: 내부 0.799 균일 |
| 2 | BM |U| > 0.05 | 경계조건 미학습 | Phase B: hard mask로 우회 |
| 3 | 설계변수 무반응 | Parametric 학습 실패 | Phase B: slit_dist 힌트 주입 |
| 4 | L_H 가중치 < 0.5 | PDE 학습 포기 | Phase B: lambda=0.01 |

In [ ]:
import sys, json
from pathlib import Path

def _find_root():
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / 'pyproject.toml').exists(): return p
        p = p.parent
    raise FileNotFoundError('Cannot find project root')

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from backend.core.pinn_model import PurePINN
from backend.training.red_flag_detector import detect_red_flags
print('Ready')

---
## Red Flag History (from training log)

In [ ]:
exp_dirs = sorted((PROJECT_ROOT / 'experiments').glob('*phase_c*'),
                  key=lambda p: p.stat().st_mtime, reverse=True)

if not exp_dirs:
    print('No experiments found.')
else:
    rf_path = exp_dirs[0] / 'red_flag_history.json'
    if rf_path.exists():
        with open(rf_path) as f:
            rf_data = json.load(f)
        
        if rf_data:
            ep = [d['epoch'] for d in rf_data]
            
            fig, axes = plt.subplots(2, 2, figsize=(14, 8))
            
            ax = axes[0,0]
            ax.plot(ep, [d['interior_cov'] for d in rf_data], 'b-o', ms=3)
            ax.axhline(y=0.05, color='red', ls='--', label='Uniform threshold')
            ax.set_title('Interior CoV (higher = better)', fontsize=10)
            ax.set_ylabel('CoV')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
            
            ax = axes[0,1]
            ax.plot(ep, [d['bm1_mean_amp'] for d in rf_data], 'g-o', ms=3, label='BM1')
            ax.plot(ep, [d['bm2_mean_amp'] for d in rf_data], 'r-o', ms=3, label='BM2')
            ax.axhline(y=0.05, color='gray', ls='--', label='Target')
            ax.set_title('BM |U| (lower = better)', fontsize=10)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
            
            ax = axes[1,0]
            ax.plot(ep, [d['design_sensitivity'] for d in rf_data], 'm-o', ms=3)
            ax.axhline(y=0.01, color='red', ls='--', label='Min threshold')
            ax.set_title('Design Sensitivity (higher = better)', fontsize=10)
            ax.set_xlabel('Epoch')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
            
            ax = axes[1,1]
            flags = [1 if d['has_red_flag'] else (0.5 if d['has_warning'] else 0) for d in rf_data]
            colors = ['#C62828' if f==1 else '#FF6F00' if f==0.5 else '#2E7D32' for f in flags]
            ax.bar(range(len(ep)), flags, color=colors)
            ax.set_title('Red Flag Status (0=OK, 0.5=Warning, 1=RED FLAG)', fontsize=10)
            ax.set_xlabel('Check #')
            ax.set_yticks([0, 0.5, 1])
            ax.set_yticklabels(['OK', 'Warning', 'RED FLAG'])
            ax.grid(True, alpha=0.2)
            
            plt.suptitle(f'Red Flag History ({exp_dirs[0].name})', fontsize=13, fontweight='bold')
            plt.tight_layout()
            plt.show()
        else:
            print('No red flag data yet.')
    else:
        print(f'No red_flag_history.json in {exp_dirs[0].name}')

---
## Live Red Flag Check (on latest checkpoint)

In [ ]:
import yaml

ckpts = sorted((PROJECT_ROOT / 'checkpoints').glob('phase_c_*.pt'),
               key=lambda p: p.stat().st_mtime, reverse=True)

if ckpts:
    latest = ckpts[0]
    print(f'Checking: {latest.name}')
    
    if exp_dirs and (exp_dirs[0] / 'config.yaml').exists():
        with open(exp_dirs[0] / 'config.yaml') as f:
            mcfg = yaml.safe_load(f)['model']
    else:
        mcfg = {'hidden_dim': 128, 'num_layers': 4, 'num_freqs': 48, 'omega_0': 30.0}
    
    model = PurePINN(**mcfg)
    ckpt = torch.load(latest, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    report = detect_red_flags(model, torch.device('cpu'))
    
    print(f'\nEpoch: {ckpt["epoch"]}')
    print(f'Interior CoV:       {report.interior_cov:.4f}')
    print(f'BM1 mean |U|:       {report.bm1_mean_amp:.4f}')
    print(f'BM2 mean |U|:       {report.bm2_mean_amp:.4f}')
    print(f'Design sensitivity: {report.design_sensitivity:.4f}')
    print()
    print(report.summary())
else:
    print('No checkpoints found.')